In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
# -------------------------
# Inputs
# -------------------------
de_paths = {
    "cluster1": "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/DE_cluster1_R_vs_NR.full.csv",
    "cluster4": "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells/DE_cluster4_R_vs_NR.full.csv",
}
outdir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/R_vs_NR_plots"  # 你也可以改成你项目里的结果目录
os.makedirs(outdir, exist_ok=True)
h5ad_path = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/05b_trajectory_revisit/adata_CD8_subset_clusters_0_1_2_4.h5ad"

# Scanpy rank_genes_groups 常见列名（你这两个文件就是这套）
COL_GENE = "names"
COL_SCORE = "scores"
COL_LFC = "logfoldchanges"
COL_P = "pvals"
COL_PADJ = "pvals_adj"

# 阈值（你可以按需要改）
PADJ_THR = 0.05
LFC_THR = 0.25

In [ ]:
# Scanpy figure dir（sc.pl.*(..., save=...) 会自动存这里）
sc.settings.figdir = outdir
sc.settings.verbosity = 2

# -----------------------
# Load adata
# -----------------------
adata = sc.read_h5ad(h5ad_path)


In [ ]:
print("X:", type(adata.X), adata.X.shape)
print("layers:", list(adata.layers.keys())[:20])
print("obs cols:", [c for c in ["Respond","leiden_T_0.6","leiden","patient"] if c in adata.obs.columns])

# 选择一个更适合画表达的 layer（优先 norm_expr，其次 log1p_norm；都没有就用 X）
adata.X = np.log1p(adata.layers["norm_expr"])

In [ ]:
CLUSTER_COL = "leiden_T_0.6"
RESP_COL = "Respond"
COL_GENE = "names"
COL_LFC  = "logfoldchanges"
COL_PADJ = "pvals_adj"

TOP_N = 25          # 每个方向取多少基因
PADJ_THR = 0.05     # 过滤阈值（可改）
LFC_THR  = 0.25     # 过滤阈值（可改）

def pick_top_genes(de_csv, top_n=TOP_N, padj_thr=PADJ_THR, lfc_thr=LFC_THR):
    df = pd.read_csv(de_csv).dropna(subset=[COL_GENE, COL_LFC, COL_PADJ]).copy()
    # 先过滤“更可信”的
    df = df[(df[COL_PADJ] < padj_thr) & (df[COL_LFC].abs() > lfc_thr)].copy()

    up = df.sort_values([COL_LFC, COL_PADJ], ascending=[False, True])[COL_GENE].head(top_n).astype(str).tolist()
    dn = df.sort_values([COL_LFC, COL_PADJ], ascending=[True,  True])[COL_GENE].head(top_n).astype(str).tolist()
    return up, dn

top_genes = {}
for cl, p in de_paths.items():
    up, dn = pick_top_genes(p)
    top_genes[cl] = {"up": up, "down": dn}
    print(cl, "up:", len(up), "down:", len(dn))

# 和 adata.var_names 取交集，避免基因不存在导致报错
def intersect_varnames(genes, ad):
    s = set(ad.var_names.astype(str))
    return [g for g in genes if g in s]

for cl in top_genes:
    top_genes[cl]["up"] = intersect_varnames(top_genes[cl]["up"], adata)
    top_genes[cl]["down"] = intersect_varnames(top_genes[cl]["down"], adata)
    print(cl, "after intersect:", len(top_genes[cl]["up"]), len(top_genes[cl]["down"]))


In [ ]:
def plot_cluster_R_NR(ad, cluster_id, genes_up, genes_down):
    ad_cl = ad[ad.obs[CLUSTER_COL].astype(str) == str(cluster_id)].copy()
    print(f"cluster {cluster_id}:", ad_cl.shape, ad_cl.obs["Respond"].value_counts().to_dict())

    genes = genes_up + genes_down
    genes = [g for g in genes if g in ad_cl.var_names]  # 再保险

    # dotplot
    sc.pl.dotplot(
        ad_cl,
        var_names=genes,
        groupby="Respond",
        standard_scale="var",   # 让不同基因可比；不喜欢可删
        swap_axes=True,
        show=True,
        save=f"_dotplot_cl{cluster_id}_RvsNR.png"
    )

    # matrixplot（更像 heatmap）
    sc.pl.matrixplot(
        ad_cl,
        var_names=genes,
        groupby="Respond",
        standard_scale="var",
        swap_axes=True,
        show=True,
        save=f"_matrixplot_cl{cluster_id}_RvsNR.png"
    )

# cluster1
plot_cluster_R_NR(
    adata,
    cluster_id="1",
    genes_up=top_genes["cluster1"]["up"],
    genes_down=top_genes["cluster1"]["down"],
)

# cluster4
plot_cluster_R_NR(
    adata,
    cluster_id="4",
    genes_up=top_genes["cluster4"]["up"],
    genes_down=top_genes["cluster4"]["down"],
)
